In [ ]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [ ]:
import mujoco_playground as mp
from mujoco_playground import registry 

from gpe.utils import acting
from gpe.utils.models import MHPolicy
from brax.envs.wrappers.training import VmapWrapper, EpisodeWrapper

import jax
import jax.numpy as jnp

import flax.nnx as nnx

In [ ]:
# registry.ALL_ENVS

seed = 0
key = jax.random.PRNGKey(seed)
rngs = nnx.Rngs(seed)

In [ ]:
task = 'HopperHop'
default_env = registry.load(task)
env = acting.wrap_env_for_training(default_env, 200)

In [ ]:
env_key = jax.random.split(key, 3)
env_state = env.reset(env_key)

In [ ]:
env_state.obs.shape

In [ ]:
a = jnp.zeros(shape=(3, env.action_size))
env_state = env.step(env_state, a)

In [ ]:
obs_dim, act_dim = env.observation_size, env.action_size
policy_model = MHPolicy(
    rngs=rngs,
    obs_dim=obs_dim,
    act_dim=act_dim,
    beta=1.0,
    hidden_size=256,
)

In [ ]:
obs = env_state.obs
init_action = env_state.info["init_action"]
act, _ = policy_model(obs, init_action, rngs())

In [ ]:
_, transitions = acting.generate_unroll(env, env_state, policy_model, rngs(), 30)

In [ ]:
transitions.reward